In [ ]:
!nvidia-smi

Mon Jan  5 12:13:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import time
import random
import argparse
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
import torchvision.transforms as transforms
from torchvision import models, datasets
from torchvision.models import resnet18
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function

In [ ]:
class SimCLRDataTransform(object):
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, sample):
        xi = self.transform(sample)
        xj = self.transform(sample)
        return xi, xj

In [ ]:
class DataSetWrapper(object):
    def __init__(self, batch_size, num_workers, val_size, input_shape, strength):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_size = val_size
        self.input_shape = input_shape
        self.strength = strength

    def get_data_loaders(self):
        data_augment = self._simclr_transform()
        train_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/Colab Notebooks/Fast Campus Image Processing/Datasets/14-02. (2) 식물 질병 분류 (Cassava Leaves) 실습 - 데이터셋/14/extraimages/0/',
                                             transform=SimCLRDataTransform(data_augment))

        train_loader, val_loader = self.get_loaders(train_dataset)
        return train_loader, val_loader

    def _simclr_transform(self):
        data_transforms: transforms.Compose

        s = self.strength
        color_jitter = transforms.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.8*s)

        data_transforms = transforms.Compose([
            transforms.RandomResizedCrop(512),
            transforms.RandomApply([color_jitter], p=0.4),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor()
        ])

        return data_transforms

    def get_loaders()


